# Nagomi — Qwen3-TTS Audio Generation

Run this notebook on **Colab with a T4 GPU** to produce all WAV files for the Nagomi conversations.

**Before you start:**
1. Click **Runtime → Change runtime type → T4 GPU**.
2. Locally, run `npm run prep:colab` in your `nagomi/` folder. That produces `nagomi_qwen_input.zip`.
3. Then **Runtime → Run all**. The notebook will prompt you to upload that zip in cell 4.

When it finishes, your browser downloads `nagomi_qwen_output.zip`. Extract that into your local `nagomi/audio/` (replace the Edge TTS MP3s). Relaunch Nagomi with `launch-nagomi.bat`.

Total runtime: typically **30–60 minutes** on T4 for 23 conversations × ~7 lines × 2 languages = 359 audio files.

## 1. Sanity-check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError('No GPU detected. Runtime → Change runtime type → T4 GPU')

## 2. Install Qwen3-TTS

The official package is `qwen-tts` on PyPI (per the Hugging Face model card). Plus `soundfile` for WAV I/O and `openai-whisper` for word-timing fallback.

In [ ]:
%pip install -q -U qwen-tts soundfile openai-whisper

## 3. Upload your input bundle

When the file picker opens, pick **`nagomi_qwen_input.zip`** (produced by `npm run prep:colab` on your local machine).

In [ ]:
import os, shutil, zipfile
from google.colab import files

# Clean previous runs.
for d in ('conversations', 'audio'):
    if os.path.isdir(d):
        shutil.rmtree(d)
for f in ('characters.json', 'generate_audio.py', 'nagomi_qwen_input.zip', 'nagomi_qwen_output.zip'):
    if os.path.exists(f):
        os.remove(f)

print('Pick nagomi_qwen_input.zip when the file dialog opens.')
uploaded = files.upload()

zip_name = next(iter(uploaded.keys()))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

print('Files in working dir:')
for f in sorted(os.listdir('.')):
    if not f.startswith('.'):
        print(' ', f)
n_conv = len([f for f in os.listdir('conversations') if f.startswith('conv_') and f.endswith('.json')])
print(f'\nUnpacked {n_conv} conversation(s).')

## 4. Generate audio

This is the slow part — the first time it runs, it downloads the Qwen3-TTS model (~3.5 GB) before producing audio. Subsequent re-runs (if the runtime survives) skip already-rendered files thanks to the script's resume support.

In [ ]:
!python generate_audio.py \
    --conversations conversations \
    --characters    characters.json \
    --audio_out     audio \
    --log           audio/generation_log.csv

## 5. Bundle and download

Zips the `audio/` folder and triggers a browser download. Extract the zip into your local `nagomi/audio/` directory, replacing the Edge TTS MP3s.

In [ ]:
import os, zipfile
from google.colab import files

OUT = 'nagomi_qwen_output.zip'
if os.path.exists(OUT):
    os.remove(OUT)

wav_count = 0
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk('audio'):
        for f in fs:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, '.')
            z.write(full, rel)
            if f.endswith('.wav'):
                wav_count += 1

size_mb = os.path.getsize(OUT) / (1024 * 1024)
print(f'Bundled {wav_count} WAV files ({size_mb:.1f} MB) into {OUT}')
files.download(OUT)

## What to do with the download

1. Move `nagomi_qwen_output.zip` into your local `nagomi/` directory.
2. Extract it. You should get `audio/conv_00001/...wav` etc. — these replace (or live alongside) the Edge TTS MP3s.
3. Run `launch-nagomi.bat`. The player tries `.wav` first, so Qwen audio wins automatically.

If you change conversations later, re-run `npm run prep:colab` and re-upload to this notebook. The script skips already-rendered files.